# 4-1 · 비·눈이 오는 시간, 대여는 평시의 몇 배인가

**⏱ 60분** · 4주차 교재 1/2 — 평균으로 비교하기 → (4-2) 회귀로 조건을 맞춰 비교하고 계획 배수 정하기

## 🎯 이번 주의 질문

공유자전거 운영팀은 시간대마다 대여가 얼마나 될지를 보고 자전거 배치와 작업 일정을 짭니다. 비나 눈이 오면 대여가 줄어드니, 예보가 있으면 그 시간의 계획을 낮춰 잡아야 합니다.

> **비·눈 예보가 있는 시간의 대여 수요를 평시의 몇 배로 계획할까요?**

너무 조금 낮추면 쓰이지 않을 자전거와 작업을 준비하게 되고, 너무 많이 낮추면 실제 대여를 받지 못합니다. 얼마나 낮출지는 지난 기록에서 정합니다.

이번 시간에는 가장 쉬운 방법인 **평균끼리 나누기**로 답을 구하고, 그 답에 날씨 말고 다른 차이가 섞여 있지 않은지 확인합니다.

**데이터:** 미국 워싱턴 D.C.의 공유자전거 Capital Bikeshare가 2011~2012년에 남긴 시간별 대여 기록에 그 시간의 날씨를 붙인 자료입니다([UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)).

**이번 시간에 반복할 패턴:** 묶어서 평균 `groupby(묶을 열)[값 열].mean()` → 두 평균 나누기(비·눈 ÷ 평시)

### 셀 표시 읽는 법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행하고 **결과만** 읽습니다. 코드를 외울 필요는 없습니다 |
| ✍️ | 직접 입력 | 보여 준 코드를 **손으로 타이핑**해서 실행합니다. 이번 주에 반복하는 핵심 패턴입니다 |
| 📝 | 과제 | 앞에서 입력한 코드를 **조금 바꿔서** 스스로 풉니다 |
| 💬 · ✅ | 결과 해설 · 체크포인트 | 숫자를 읽는 법, 내 결과가 맞는지 확인할 값 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 공유자전거 기록을 내려받아 표 h로 읽습니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
h = load("bike")
print("준비 완료!")

✅ `준비 완료!`가 나오면 됩니다.

---
## 1. 한 행은 무엇이고, 기간은 얼마인가

표를 받으면 크기와 앞부분부터 봅니다. `.shape`는 (행 수, 열 수), `.head()`는 앞의 다섯 행입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(h.shape)
h.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 17,379행 × 17열입니다. 첫 행은 2011년 1월 1일 0시(`hr` 0), 둘째 행은 같은 날 1시입니다. **한 행 = 한 날짜의 한 시간대**이고, 대여 건수 `cnt`는 그 한 시간 동안 일어난 대여의 수입니다.

이번 주에 쓰는 열은 다음과 같습니다. 이 열들에 빈 값은 없습니다. 보유 자전거 수나 정류장별 재고는 이 자료에 없습니다.

| 열 | 뜻 |
|---|---|
| `dteday`, `hr` | 날짜, 시간대(0~23) |
| `cnt` | 전체 대여 건수. 회원 `registered`와 비회원 `casual`의 합 |
| `weathersit` | 날씨. 1 맑음·구름, 2 안개·흐림, 3 약한 비·눈, 4 강한 비·눈 |
| `temp`, `hum`, `windspeed` | 기온·습도·풍속을 0~1 사이로 바꿔 놓은 값 |
| `mnth`, `workingday`, `yr` | 월, 근무일 여부(주말·휴일이면 0), 연도(2011년 0, 2012년 1) |

기간도 확인합니다. `.min()`·`.max()`는 가장 이른 날과 늦은 날, `.nunique()`는 서로 다른 값의 개수입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print("첫날:", h["dteday"].min())
print("마지막 날:", h["dteday"].max())
print("날짜 수:", h["dteday"].nunique())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 2011년 1월 1일부터 2012년 12월 31일까지 **731일**입니다.
- 731일 × 24시간 = 17,544시간인데 행은 17,379개입니다. **165시간은 기록이 없습니다.**
- 이 시간을 대여 0으로 채우지 않고, 기록된 시간끼리만 비교합니다.

---
## 2. 비·눈이 온 시간은 몇 시간인가

비·눈 시간이 너무 적으면 비교가 흔들리므로 규모부터 봅니다. `.value_counts()`는 값마다 개수를 세고, `.sort_index()`는 날씨 코드 순서로 놓습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
h["weathersit"].value_counts().sort_index()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 맑음·구름(1) 11,413시간, 안개·흐림(2) 4,544시간, 약한 비·눈(3) 1,419시간, 강한 비·눈(4)은 **3시간**뿐입니다.
- 3시간으로는 강한 비·눈을 따로 비교할 수 없습니다. 그래서 3과 4를 묶어 **비·눈 시간**, 1과 2를 **평시**로 둡니다.

비·눈이면 1, 평시면 0인 열 `rain`을 만듭니다. `h["weathersit"] >= 3`은 행마다 참(True)·거짓(False)을 돌려주고, `.astype(int)`는 이것을 1·0으로 바꿉니다. `h["새 열"] = ...`로 쓰면 표에 새 열이 붙습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
h["rain"] = (h["weathersit"] >= 3).astype(int)
h["rain"].value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 0이 15,957, 1이 1,422이면 맞습니다.

💬 평시 15,957시간, 비·눈 1,422시간(3과 4를 합친 수)입니다. 비·눈 시간은 전체의 약 8%입니다.

---
## 3. 대여는 시간대마다 얼마나 다른가

비·눈의 차이를 보기 전에, 날씨와 상관없이 하루 동안 대여가 얼마나 오르내리는지 봅니다.

**묶어서 평균** — 이번 주에 가장 많이 쓰는 패턴입니다. 읽는 순서: `h`를 `hr`로 묶어서 → `cnt` 열의 → 평균. 결과를 `hourly`에 담고, `.round(1)`로 소수 첫째 자리까지 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
hourly = h.groupby("hr")["cnt"].mean()
hourly.round(1)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


24개 숫자는 그림으로 보면 한눈에 들어옵니다.

In [ ]:
# ▶ 그대로 실행하세요 — 시간대별 평균 대여 그림
plt.figure(figsize=(8, 4))
plt.plot(hourly.index, hourly.values, marker="o")
plt.xlabel("Hour of day")
plt.ylabel("Mean rentals per hour")
plt.xticks(range(0, 24, 3))
plt.show()

💬 **결과 해설** — 새벽 4시의 평균은 약 **6.4건**, 오후 5시는 약 **461.5건**입니다.
- 아침 8시와 오후 5~6시에 대여가 몰리고 새벽에는 거의 없습니다. 같은 한 시간이라도 **70배 넘게** 다릅니다.
- 그래서 운영팀은 시간대마다 계획을 따로 세웁니다. 비·눈 예보가 있을 때 "몇 건 줄인다"로 정하면 새벽 4시와 오후 5시에 같은 건수를 뺄 수 없습니다.
- 모든 시간대의 계획에 똑같이 곱할 수 있는 **"평시의 몇 배"**로 정하는 것이 쓰기 좋습니다. 이 배수가 이번 주에 구할 값입니다.

---
## 4. 평균끼리 나누면 몇 배인가

가장 쉬운 답부터 구합니다. 비·눈 시간과 평시의 평균 대여를 구해 나눕니다.

3절과 같은 **묶어서 평균** 패턴인데, 이번에는 `rain`으로 묶습니다. `.agg(["mean", "count"])`는 평균과 개수를 한 번에 구합니다. 평균이 몇 시간에서 나왔는지도 함께 보려는 것입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
cmp = h.groupby("rain")["cnt"].agg(["mean", "count"])
cmp.round(1)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 평시(0)는 평균 196.4건(15,957시간), 비·눈(1)은 111.5건(1,422시간)입니다.

**두 평균 나누기** — `cmp.loc[행, 열]`로 표에서 값 하나를 꺼냅니다. 비·눈(행 1)의 평균을 평시(행 0)의 평균으로 나눕니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
cmp.loc[1, "mean"] / cmp.loc[0, "mean"]
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `0.5676...`이 나오면 맞습니다.

💬 **결과 해설** — 비·눈 시간의 대여는 평시의 **약 0.57배**입니다. 이대로라면 "비·눈 예보가 있으면 평시 계획에 0.57을 곱한다"가 답입니다.

그런데 이 값은 두 집단의 모든 시간을 한꺼번에 평균해 나눈 것입니다. 만약 비·눈이 대여가 적은 새벽이나 추운 날에 몰려 왔다면, 0.57배에는 날씨 말고 **시간대·기온의 차이까지** 들어 있습니다. 5·6절에서 확인합니다.

---
## 5. 같은 시간대끼리 비교하면?

### 비·눈이 특정 시간대에 몰려 왔나

시간대마다 기록된 시간 중 비·눈 시간의 비율을 봅니다. `rain`은 0과 1이므로 **평균이 곧 1의 비율**입니다. 3절과 같은 패턴에서 `cnt` 대신 `rain`의 평균을 구합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
h.groupby("hr")["rain"].mean().round(3)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 비·눈 비율은 가장 낮은 11시 0.065에서 가장 높은 17시 0.104 사이입니다.
- 비·눈은 시간대에 **거의 고르게** 왔고, 대여가 많은 저녁 퇴근 시간에 조금 더 많았습니다.
- 대여가 적은 새벽에 몰리지는 않았습니다.

### 같은 시간대 안에서는 몇 배인가

시간대와 `rain` 두 열로 함께 묶어 평균을 구하고, 비·눈 여부를 열로 펼친 표 `by_hour`를 만듭니다. 두 열로 묶어 펼치는 코드는 이번 주에 따로 익히지 않으므로 실행만 합니다. 열 0은 평시, 열 1은 비·눈입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 시간대 × 비·눈 평균 대여 표
by_hour = h.groupby(["hr", "rain"])["cnt"].mean().unstack()
by_hour.round(1)

4절과 같은 **두 평균 나누기**를 시간대마다 합니다. 열 1을 열 0으로 나누면 24개 시간대의 배수가 한 번에 나옵니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
hour_ratio = by_hour[1] / by_hour[0]
hour_ratio.round(2)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


막대는 시간대별 배수, 점선은 4절의 평균끼리의 배수입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 시간대별 배수 그림
plt.figure(figsize=(8, 4))
plt.bar(hour_ratio.index, hour_ratio.values)
plt.axhline(cmp.loc[1, "mean"] / cmp.loc[0, "mean"], linestyle="--", color="gray")
plt.xlabel("Hour of day")
plt.ylabel("Rain / dry mean rentals")
plt.xticks(range(0, 24, 3))
plt.show()

💬 **결과 해설** — 새벽 4시와 오후 5시를 `by_hour`와 `hour_ratio`에서 찾아 비교해 봅시다.

| 시간대 | 평시 | 비·눈 | 차이 | 배수 |
|---|---|---|---|---|
| 새벽 4시 | 6.5건 | 4.7건 | 1.8건 적음 | 0.72 |
| 오후 5시 | 483.0건 | 276.0건 | 207.0건 적음 | 0.57 |

- **건수로 보면** 줄어든 폭이 1.8건과 207건으로 100배 넘게 다릅니다.
- **배수로 보면** 모든 시간대가 0.40(11시)~0.72(4시·6시) 사이에 있습니다. 그래서 비·눈의 차이는 "몇 건 적다"가 아니라 **"평시의 몇 배"**로 읽습니다.
- 시간대별 배수가 점선(0.57배) 위아래에 흩어져 있습니다. 평균끼리의 배수가 시간대 구성 때문에 크게 틀어지지는 않았습니다.

---
## 6. 기온과 바람은 같았나

추운 시간에는 날씨와 상관없이 대여가 적을 수 있습니다. 비·눈 시간이 평시보다 추웠다면 0.57배에는 기온의 차이도 들어 있습니다.

기온 `temp`는 0~1로 바꿔 놓은 값이라, 원자료의 정의대로 47을 곱하고 8을 빼 섭씨 `temp_c`로 되돌립니다.

In [ ]:
# ▶ 그대로 실행하세요 — 섭씨 기온 열 temp_c
h["temp_c"] = h["temp"] * 47 - 8
h[["temp", "temp_c"]].head(3)

**묶어서 평균**을 한 번 더 씁니다. `rain`으로 묶고 `temp_c`와 `windspeed` 두 열의 평균을 함께 구합니다. 열 여러 개는 `[["열1", "열2"]]`처럼 대괄호 안에 리스트로 고릅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
h.groupby("rain")[["temp_c", "windspeed"]].mean().round(3)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 평균 기온은 평시 15.5도, 비·눈 13.4도입니다. 풍속은 0.188과 0.219입니다.
- 비·눈 시간은 평시보다 **약 2.1도 춥고 바람이 조금 셌습니다.** 0.57배에는 "추워서 덜 탄" 몫이 섞였을 수 있습니다.
- 월, 근무일 여부, 연도도 대여 규모와 관련될 수 있습니다. 이 조건들을 5절처럼 표로 나눠 맞추려면 시간대 24 × 월 12 × … 로 칸이 너무 잘게 쪼개집니다.
- 여러 조건을 한꺼번에 맞추는 도구가 **회귀분석**입니다. 다음 노트북(4-2)에서 합니다.

---
## 7. 실습 과제 (15분)

### 📝 과제 1 · 흐림은 맑음의 몇 배인가

평시에는 맑음·구름(1)과 안개·흐림(2)을 함께 묶었습니다. 흐림만으로도 비·눈만큼 대여가 줄었다면 이렇게 묶으면 안 됩니다. 날씨 코드 `weathersit`별 평균 대여와 시간 수를 구하고, 흐림(2)의 평균이 맑음(1)의 몇 배인지 계산하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: 4절의 첫 셀에서 묶는 열 rain을 weathersit로 바꾸고, 결과를 wcmp에 담습니다.
# 힌트 2: 나누기는 4절의 둘째 셀에서 꺼내는 행 번호만 바꿉니다(비·눈 1 → 흐림 2, 평시 0 → 맑음 1).


✏️ **나의 답:** 흐림은 맑음의 몇 배인가요? 비·눈과 비교하면, 흐림을 평시에 묶은 것이 괜찮아 보이나요? (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 월마다 비·눈이 고르게 왔나

5절에서 시간대마다 비·눈 비율을 보았습니다. 같은 방법으로 **월** `mnth`마다 비·눈 비율과 평균 대여를 구하세요. 대여가 많은 달에 비·눈이 많았나요, 적었나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 5절 첫 셀에서 묶는 열 hr을 mnth로 바꿉니다. 6절처럼 rain과 cnt 두 열을 리스트로 함께 고르면 한 표로 볼 수 있습니다.


✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기

4절의 0.57배를 그대로 계획에 쓴다면 무엇이 걱정되나요? 5·6절과 과제 2의 결과를 근거로 한두 문장으로 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 한 행의 의미 | 한 날짜의 한 시간대. 731일 중 기록된 17,379시간(165시간은 기록 없음) |
| 비·눈 시간 | 날씨 3·4, 1,422시간. 평시는 날씨 1·2, 15,957시간 |
| 평균끼리의 배수 | 비·눈 111.5건 ÷ 평시 196.4건 = **약 0.57배** |
| 건수보다 배수 | 시간대마다 대여가 70배 넘게 달라도 비·눈의 배수는 0.40~0.72 사이 |
| 섞였을 수 있는 차이 | 시간대는 고르게 섞였지만, 비·눈 시간은 약 2도 춥고 바람이 셌다 → 조건을 맞춘 비교가 필요하다 |
| 반복한 패턴 | `h.groupby(묶을 열)[값 열].mean()` → 비·눈 ÷ 평시 |

다음: **4-2 · 조건을 맞춰도 절반인가 — 회귀로 구한 배수와 계획**